IKEA Gifting RFM segmentation

In [1]:
import pandas as pd

df = pd.read_csv("retail_full2.csv",
                 dtype={"invoice": str, "stockcode": str, "customer_id": str},
                 low_memory=False)

parsed = pd.to_datetime(df['invoicedate'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
df["invoicedate"] = parsed
 
print(df.shape)   # (1033036, 9)
df.info()

(1033036, 9)
<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   invoice      1033036 non-null  str           
 1   stockcode    1033036 non-null  str           
 2   description  1028761 non-null  str           
 3   quantity     1033036 non-null  int64         
 4   invoicedate  1033036 non-null  datetime64[us]
 5   Price        1033036 non-null  float64       
 6   customer_id  797885 non-null   str           
 7   country      1033036 non-null  str           
 8   sourcesheet  1033036 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(6)
memory usage: 70.9 MB


Filter and calculate recency, frequency and monetary


In [19]:
rfm_data = df[(df['customer_id'].notna()) & (df['quantity'] > 0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')

rfm_data['total_price'] = rfm_data['quantity'] * rfm_data['Price']

rfm_data = rfm_data.groupby('customer_id').agg(
    recency=('invoicedate', lambda x: (snapshot_date - x.max()).days),
    frequency = ('invoice', 'nunique'),
    monetary = ('total_price', 'sum')
).reset_index()


Score customer and assign business segmentation 

In [21]:
rfm_data['R_Score'] = pd.qcut(rfm_data['recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm_data['F_Score'] = pd.qcut(rfm_data['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_data['M_Score'] = pd.qcut(rfm_data['monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4: 
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:          
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:          
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:          
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:          
        return 'Needs Attention'
    else:                                            
        return 'Lost'

rfm_data['Segment'] = rfm_data.apply(segment_customer, axis=1)
rfm_data['IsChurned'] = rfm_data['recency'] > 90
 

In [22]:
# merge rfm results and back to transaction and export

df = df[(df['customer_id'].notna()) & (df['quantity'] > 0)].copy()
df['InvoiceMonth'] = df['invoicedate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['quantity'] * df['Price']

df_final = df.merge(rfm_data[['customer_id', 'recency', 'frequency', 'monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']], on='customer_id', how='left')
df_final.to_excel("ikea_rfm_results.xlsx", index=False)